# How to add a background to a map
In this notebook we present 2 main ways of having a background for a map:
1. Using an image from a geoTIFF file
2. Using tiles from a basemap, such as OpenStreetMap.

In [ ]:
import Pkg
Pkg.activate("../")
Pkg.instantiate()

In [ ]:
using GeoMakie
using Rasters
using ArchGDAL
using CairoMakie

## Use a geoTIFF
The file can be optained for exemple from the [Copernicus Browser](https://browser.dataspace.copernicus.eu/) or from NASA [WorldView](https://worldview.earthdata.nasa.gov/).

In [ ]:
datadir = "../data/"
datafile = joinpath(datadir, "2026-09-25-00_00_2026-09-25-23_59_Sentinel-2_L1C_True_color.tiff")

### Read coordinates and raster

In [ ]:
raster_sentinel2 = Raster(datafile)
lon = lookup(raster_sentinel2, X);
lat = lookup(raster_sentinel2, Y);
data = Array(raster_sentinel2.data);

#### Extract the bounding box from the coordinates

In [ ]:
ext = Rasters.extent(raster_sentinel2)
bbox = (ext.X[1], ext.X[2], ext.Y[1], ext.Y[2])

### Plot using `surface`

In [ ]:
rgb = RGB.(data[:, :, 1] ./ 255, data[:, :, 2] ./ 255, data[:, :, 3] ./ 255);

In [ ]:
gridoptions = Dict(
    :source => "+proj=longlat +datum=WGS84",
    :limits => bbox,
    :dest   => "+proj=robin",
    :xgridstyle => :dash,
    :ygridstyle => :dash,
    :xgridcolor => :gray70,
    :ygridcolor => :gray70,
    :xticks => 2.:0.5:4.,
    :yticks => 38.75:0.25:40.25,
    :titlesize => 20,
)

In [ ]:
fig = Figure()
ga = GeoAxis(fig[1, 1]; gridoptions...)

surface!(ga, lon, lat, zeros(length(lon), length(lat));
         color = rgb, shading = NoShading)

lines!(ga, GeoMakie.coastlines(); color = :yellow, linewidth = 0.5)
display(fig)

In [ ]:
fig = Figure()
ga = GeoAxis(fig[1, 1]; gridoptions...)

meshimage!(ga, (ext.X[1], ext.X[2]), (ext.Y[1], ext.Y[2]), reverse(rgb, dims = 2), npoints = 200)
lines!(ga, GeoMakie.coastlines(10); color = :yellow, linewidth = 0.5)
display(fig)

## Tiles

In [ ]:
using MapTiles
using TileProviders
using Rasters
using Downloads
using Images
using Extents
using Proj
using GeoFormatTypes
using DimensionalData.Lookups

In [ ]:
zoom_level = 9

In [ ]:
osmfile = joinpath(datadir, "openstreetmap_mallorca.tif")

In [ ]:
function download_osm(provider::Provider, bbox::Extent, path::AbstractString; zoom::Int64 = 11, max_tiles::Int64 = 100)

    tilegrid = TileGrid(bbox, zoom, MapTiles.wgs84)
    length(tilegrid) > max_tiles && error("$(length(tilegrid)) tiles needed; lower zoom or raise max_tiles")

    tx = extrema(t.x for t in tilegrid)
    ty = extrema(t.y for t in tilegrid)                      # tile y grows southwards
    ntx, nty = tx[2] - tx[1] + 1, ty[2] - ty[1] + 1

    mosaic = zeros(UInt8, ntx * 256, nty * 256, 3)
    headers = ["User-Agent" => "Julia-OSM-tile-cache/1.0 (personal use)"]
    for tile in tilegrid
        io = IOBuffer()
        Downloads.download(TileProviders.geturl(provider, tile.x, tile.y, tile.z), io; headers)
        seekstart(io)
        img = RGB.(PNGFiles.load(io))
        i, j = tile.x - tx[1], tile.y - ty[1]
        mosaic[i*256+1:(i+1)*256, j*256+1:(j+1)*256, :] .=
            permutedims(rawview(channelview(img)), (3, 2, 1))
        sleep(0.1)
    end
    # data = reverse(mosaic, dims = 2)                         # y ascending
    data = mosaic
    ext = Extents.extent(tilegrid, MapTiles.web_mercator)    # exact mosaic extent in metres
    (x0, x1), (y0, y1) = ext.X, ext.Y
    nx, ny = size(data, 1), size(data, 2)
    dx, dy = (x1 - x0) / nx, (y1 - y0) / ny
    dims = (
        X(Projected(range(x0, step = dx, length = nx);
            sampling = Intervals(Start()), order = ForwardOrdered(),
            span = Regular(dx), crs = EPSG(3857))),
        Y(Projected(range(y0, step = dy, length = ny);
            sampling = Intervals(Start()), order = ForwardOrdered(),
            span = Regular(dy), crs = EPSG(3857))),
        Rasters.Band(1:3),
    )
    write(path, Raster(data; dims))
    return path
end

In [ ]:
bbox = Extent(X = (3.726, 4.381), Y = (39.7688, 40.1188));
regionname = "Menorca"
zoomlevel = 12
basemapfile = joinpath(datadir, "$(regionname)_$(replace(pp.options[:name], "." => "-"))_$(zoomlevel).tif")

if isfile(basemapfile) 
    @info("File already downloaded") 
else
    download_osm(OpenStreetMap(), bbox, basemapfile; zoom = zoomlevel)
end

In [ ]:
bbox.X

In [ ]:
gridoptions = Dict(
    :source => "+proj=longlat +datum=WGS84",
    :limits => (bbox.X[1], bbox.X[2], bbox.Y[1], bbox.Y[2]),
    :dest   => "+proj=robin",
    :xgridstyle => :dash,
    :ygridstyle => :dash,
    :xgridcolor => :gray70,
    :ygridcolor => :gray70,
    :xticks => 2.:0.5:4.,
    :yticks => 38.75:0.25:40.25,
    :titlesize => 20,
)

In [ ]:
### 
raster_osm = Raster(basemapfile)
lon_osm = lookup(raster_osm, X);
lat_osm = lookup(raster_osm, Y);
data_osm = Array(raster_osm.data);

osm4326 = Rasters.resample(raster_osm; crs = EPSG(4326), method = :near)   # :near keeps UInt8 colours
(xmin, xmax), (ymin, ymax) = Rasters.bounds(osm4326, X), Rasters.bounds(osm4326, Y)

A = Array(osm4326.data)
A = coalesce.(A, 0x00)                # missing -> 0, now plain UInt8
rgb = RGB.(A[:, :, 1] ./ 255, A[:, :, 2] ./ 255, A[:, :, 3] ./ 255)
Rasters.isreverse(dims(osm4326, Y)) && (rgb = reverse(rgb, dims = 2))

In [ ]:
fig = Figure(size=(1000, 1000))
ga = GeoAxis(fig[1, 1]; gridoptions...)

meshimage!(ga, (xmin, xmax), (ymin, ymax), reverse(rgb, dims = 2), npoints = 2000)
lines!(ga, GeoMakie.coastlines(10); color = :yellow, linewidth = 0.5)
save("../figures/test_osm_$(zoom_level).png", fig)